In [7]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm

# === CONFIG ===
CLEAN_DIR    = Path.home() / "Downloads" / "ASR_Project_Group_8" / "tlt_english_clean"
METADATA_CSV = CLEAN_DIR / "metadata.csv"          # output van je cleaning-notebook
OUTPUT_BASE  = str(Path.home() / "Documents" / "tlt2021_augmented")
os.makedirs(OUTPUT_BASE, exist_ok=True)

SR            = 16000          # Whisper-vereiste sample rate
SPEED_FACTORS = [0.9, 1.1]     # speed perturbation 
MAX_DURATION  = 30.0           # Whisper-limiet (sec) — langere clips overslaan
np.random.seed(42)

print("Metadata bestaat:", METADATA_CSV.exists())
print("Output map:", OUTPUT_BASE)

Metadata bestaat: True
Output map: /Users/liekebugel/Documents/tlt2021_augmented


In [8]:
meta = pd.read_csv(METADATA_CSV)
print("Kolommen:", list(meta.columns))

meta["utt_id"] = meta["audio_path"].apply(lambda p: Path(p).stem)

df_train = meta[meta["split"] == "train"].reset_index(drop=True)
df_test  = meta[meta["split"] == "test"].reset_index(drop=True)

print(f"\nTrain: {len(df_train)} utts, {df_train['speaker_id'].nunique()} sprekers, "
      f"{df_train['duration'].sum()/3600:.2f} uur")
print(f"Test : {len(df_test)} utts, {df_test['speaker_id'].nunique()} sprekers, "
      f"{df_test['duration'].sum()/3600:.2f} uur")

# --- Checks zodat je niet stilletjes met lege data verder gaat ---
assert len(df_train) > 0, "df_train is leeg! Klopt de 'split'-kolom (train/test)?"
assert len(df_test)  > 0, "df_test is leeg! Gebruikt je csv wel 'test' (niet 'dev')?"

missing = [p for p in df_train["audio_path"].head(20) if not os.path.exists(p)]
assert not missing, f"Audio-paden bestaan niet, bijv.: {missing[:3]}"
print("\nChecks OK: train/test gevuld en audio-paden bestaan.")

Kolommen: ['audio_path', 'sentence', 'speaker_id', 'duration', 'cefr', 'source', 'split', 'is_split']

Train: 10254 utts, 2389 sprekers, 27.04 uur
Test : 2683 utts, 594 sprekers, 7.15 uur

Checks OK: train/test gevuld en audio-paden bestaan.


In [9]:
# === 10%-SUBSET (spreker-niveau, gestratificeerd op CEFR) ===
# Doel: 1 vaste subset van 10% van de TRAIN-sprekers, gebruikt voor BEIDE condities.
#   - origineel-only  = deze 10%
#   - augmented       = DEZELFDE 10%, maar met speed-augmentatie erbij (~3x zoveel audio)
# Test blijft VOLLEDIG (nooit subsetten, nooit augmenteren).
SUBSET_FRAC = 0.10

# Per spreker het meest voorkomende CEFR-niveau (voor stratificatie)
spk_cefr = (df_train.groupby("speaker_id")["cefr"]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else "unknown")
            .reset_index())

# Binnen elk CEFR-niveau 10% van de sprekers trekken (reproduceerbaar)
keep_speakers = []
for level, grp in spk_cefr.groupby("cefr"):
    n = max(1, round(len(grp) * SUBSET_FRAC))
    keep_speakers += list(grp["speaker_id"].sample(n=n, random_state=42))
keep_speakers = set(keep_speakers)

df_train_10 = df_train[df_train["speaker_id"].isin(keep_speakers)].reset_index(drop=True)

print(f"Sprekers : {df_train['speaker_id'].nunique()} -> {df_train_10['speaker_id'].nunique()}")
print(f"Utts     : {len(df_train)} -> {len(df_train_10)}")
print(f"Uur      : {df_train['duration'].sum()/3600:.2f} -> {df_train_10['duration'].sum()/3600:.2f}")
print("\nCEFR-verdeling 10%-subset:")
print(df_train_10["cefr"].value_counts())

Sprekers : 2389 -> 239
Utts     : 10254 -> 1079
Uur      : 27.04 -> 2.85

CEFR-verdeling 10%-subset:
cefr
A1    462
A2    288
B1    189
Name: count, dtype: int64


In [10]:
def speed_perturb(y, sr, factor):
    """Speed perturbation via resampling (Ko et al., 2015).
    factor 0.9 -> langzamer/lager (langer);  1.1 -> sneller/hoger (korter)."""
    new_sr = int(sr / factor)
    return librosa.resample(y, orig_sr=sr, target_sr=new_sr)

def safe_name(s):
    """Veilige map-/bestandsnaam van bijv. speaker_id."""
    return "".join(c if (c.isalnum() or c in "-_.") else "_" for c in str(s))

def write_clip(spk_dir, clip_id, audio, sentence):
    """Schrijf één .wav + bijbehorende .txt."""
    wav_path = os.path.join(spk_dir, f"{clip_id}.wav")
    txt_path = os.path.join(spk_dir, f"{clip_id}.txt")
    sf.write(wav_path, audio, SR, subtype="PCM_16")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(str(sentence))
    return wav_path, txt_path

In [11]:
def build_set(df, run_name, factors, write_original=True):
    """
    Schrijft per clip een .wav + .txt, gesorteerd per speaker_id:
        OUTPUT_BASE/<run_name>/<speaker_id>/<utt_id>.wav (+ .txt)
    factors=[]            -> alleen originelen (origineel-only set)
    factors=[0.9, 1.1]    -> originelen + speed-perturbaties
    Geeft een DataFrame terug met alle weggeschreven rijen (incl. txt_path).
    """
    out_dir = os.path.join(OUTPUT_BASE, run_name)
    os.makedirs(out_dir, exist_ok=True)
    rows, n_skipped = [], 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc=run_name):
        y, _ = librosa.load(row["audio_path"], sr=SR)
        sentence = row["sentence"]
        spk_dir = os.path.join(out_dir, safe_name(row["speaker_id"]))
        os.makedirs(spk_dir, exist_ok=True)

        # Origineel (geresampled naar 16 kHz) meeschrijven
        if write_original:
            wav_path, txt_path = write_clip(spk_dir, row["utt_id"], y, sentence)
            orig = row.to_dict()
            orig["augmentation"] = "original"
            orig["audio_path"]   = wav_path
            orig["txt_path"]     = txt_path
            rows.append(orig)

        # Speed-perturbaties
        for factor in factors:
            y_aug   = speed_perturb(y, SR, factor)
            new_dur = len(y_aug) / SR
            if new_dur > MAX_DURATION:        # >30s na verlangzaming -> overslaan
                n_skipped += 1
                continue
            new_id = f"{row['utt_id']}_speed{factor}"
            wav_path, txt_path = write_clip(spk_dir, new_id, y_aug, sentence)
            new_row = row.to_dict()
            new_row["utt_id"]       = new_id
            new_row["audio_path"]   = wav_path
            new_row["txt_path"]     = txt_path
            new_row["augmentation"] = f"speed{factor}"
            new_row["duration"]     = round(new_dur, 3)   # duur verandert bij speed
            rows.append(new_row)

    if n_skipped:
        print(f"  ! {n_skipped} clips overgeslagen (>30s na verlangzaming)")
    return pd.DataFrame(rows)

In [12]:
# 1) Alleen originele data (baseline-training) — 10% subset
df_orig = build_set(df_train_10, run_name="tlt_original_only_10pct",
                    factors=[], write_original=True)
print(f"\nOrigineel-only (10%): {len(df_orig)} utts, {df_orig['duration'].sum()/3600:.2f} uur")

# 2) Originelen + speed perturbation (augmented training) — DEZELFDE 10% subset
df_aug = build_set(df_train_10, run_name="tlt_speed_augmented_10pct",
                   factors=SPEED_FACTORS, write_original=True)
print(f"\nSpeed-augmented (10%): {len(df_aug)} utts, {df_aug['duration'].sum()/3600:.2f} uur")
print(df_aug["augmentation"].value_counts())

# 3) Test-set (origineel, VOLLEDIG, NOOIT augmenteren)
df_test_out = build_set(df_test, run_name="tlt_test",
                        factors=[], write_original=True)
print(f"\nTest: {len(df_test_out)} utts, {df_test_out['duration'].sum()/3600:.2f} uur")

tlt_original_only_10pct:   0%|          | 0/1079 [00:00<?, ?it/s]


Origineel-only (10%): 1079 utts, 2.85 uur


tlt_speed_augmented_10pct:   0%|          | 0/1079 [00:00<?, ?it/s]


Speed-augmented (10%): 3237 utts, 8.60 uur
augmentation
original    1079
speed0.9    1079
speed1.1    1079
Name: count, dtype: int64


tlt_test:   0%|          | 0/2683 [00:00<?, ?it/s]


Test: 2683 utts, 7.15 uur


In [13]:
df_orig.to_csv(os.path.join(OUTPUT_BASE, "tlt_original_only_10pct.csv"), index=False)
df_aug.to_csv(os.path.join(OUTPUT_BASE, "tlt_speed_augmented_10pct.csv"), index=False)
df_test_out.to_csv(os.path.join(OUTPUT_BASE, "tlt_test.csv"), index=False)
print("Index-CSV's opgeslagen in:", OUTPUT_BASE)

Index-CSV's opgeslagen in: /Users/liekebugel/Documents/tlt2021_augmented


In [15]:
# wav/txt-paren kloppen?
for name in ["tlt_original_only_10pct", "tlt_speed_augmented_10pct", "tlt_test"]:
    run_dir = os.path.join(OUTPUT_BASE, name)
    n_wav = len(glob.glob(os.path.join(run_dir, "*", "*.wav")))
    n_txt = len(glob.glob(os.path.join(run_dir, "*", "*.txt")))
    print(f"{name:22s}: {n_wav} wav | {n_txt} txt  (moet gelijk zijn)")
    assert n_wav == n_txt, f"wav/txt mismatch in {name}!"

# Alle clips <= 30s?
assert (df_aug["duration"]  <= MAX_DURATION + 0.01).all(), "Clip >30s in augmented!"
assert (df_orig["duration"] <= MAX_DURATION + 0.01).all(), "Clip >30s in origineel!"

# Geen speaker-overlap train/test
overlap = set(df_train["speaker_id"]) & set(df_test["speaker_id"])
print("Speaker-overlap train/test:", len(overlap), "(moet 0 zijn)")

# Steekproef: speelt af / leest txt
r = df_aug.sample(1, random_state=0).iloc[0]
y, sr = librosa.load(r["audio_path"], sr=None)
print(f"\nVoorbeeld: {r['augmentation']} | spk={r['speaker_id']} | {len(y)/sr:.2f}s")
print(f"  txt: {open(r['txt_path'], encoding='utf-8').read()[:60]}...")
print("\nOK: per-clip wav+txt augmentatie geslaagd.")

tlt_original_only_10pct: 1079 wav | 1079 txt  (moet gelijk zijn)
tlt_speed_augmented_10pct: 3237 wav | 3237 txt  (moet gelijk zijn)
tlt_test              : 2683 wav | 2683 txt  (moet gelijk zijn)
Speaker-overlap train/test: 0 (moet 0 zijn)

Voorbeeld: original | spk=spk_2040120 | 7.01s
  txt: are you...

OK: per-clip wav+txt augmentatie geslaagd.
